# Session Sync Analysis

Visualizes results created by `sync_session_etl.py`.

In [ ]:
import duckdb, glob, pandas as pd, plotly.express as px

In [ ]:
db_candidates = sorted(glob.glob('../../data/mydb2024-25*.duckdb'))
db_path = db_candidates[-1] if db_candidates else '../data/mydb.duckdb'
print(f'Using DuckDB database: {db_path}')
con = duckdb.connect(db_path)

In [ ]:
# Load sync log table
try:
    df_log = con.execute('SELECT * FROM fixtures_session_sync_log').df()
except Exception:
    df_log = pd.DataFrame()
df_log.head() if not df_log.empty else 'fixtures_session_sync_log is empty or missing' 

In [ ]:
# Success vs fail
if not df_log.empty and 'status' in df_log.columns:
    display(df_log['status'].value_counts())
    px.bar(df_log['status'].value_counts().reset_index().rename(columns={'index':'status','status':'count'}), x='status', y='count', title='Sync Results')
else:
    print('No status column present.')

In [ ]:
# Similarity scores (if available)
if not df_log.empty and 'similarity_score' in df_log.columns:
    px.histogram(df_log.dropna(subset=['similarity_score']), x='similarity_score', nbins=20, title='Similarity score distribution')
else:
    print('No similarity_score column present.')